# lidarfly — LiDAR 1080빔 × 3프레임 + 초파리 커넥톰 SAC

맵 없이 **LiDAR 1080빔, FOV 270°, 40 Hz, 현재+과거 2프레임** → 커넥톰 DN 48 → 선형 읽기 → 조향·속도 (2~7 m/s).
actor 에 CNN·MLP 은닉층 없음. critic 만 레이싱라인 privileged 27 + MLP.

**보상 가중치와 DN 극성(부호)은 임의값이다.** 출처는 `lidarfly/README.md`.

**순서:** ① 드라이브·clone·설치 → ② 테스트 → ③ 스모크(선택) → ④ 본 학습 → ⑤ 평가 → ⑥ TensorBoard.

**세션이 끊기면:** ① 과 ④ 만 다시. `last_model.zip` 은 드라이브에 있고 `--resume auto` 가 이어서 학습한다.
처음 한 번: `런타임 → 런타임 유형 변경 → T4 GPU`.


## ① 드라이브 마운트 · 레포 clone · pip 설치

브랜치 `feat/mapless40-asym-sac`. 체크포인트는 구글 드라이브 `lidarfly_runs/v1` (세션이 죽어도 남음).


In [ ]:
!nvidia-smi -L || echo "GPU 없음 → 런타임 유형을 T4 GPU 로 바꾸세요"
!python --version


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/lidarfly_runs/v1'
import os; os.makedirs(RUN_DIR, exist_ok=True); print(RUN_DIR)

V_MAX = 7.0   # 사용자 지정 최고 속도 [m/s]. 학습·평가가 같은 값을 써야 함


In [ ]:
%cd /content
!test -d 2027_F1tenth_test || git clone -q --depth 1 -b feat/mapless40-asym-sac https://github.com/tkddn647-ship-it/2027_F1tenth_test
%cd /content/2027_F1tenth_test
!git pull -q origin feat/mapless40-asym-sac
!git log --oneline -1
!pip -q install "stable-baselines3>=2.3" "gymnasium>=0.29" scipy pyyaml pillow tensorboard


## ② 테스트

torch·stable-baselines3 가 있으면 **10/10**. 없으면 numpy 8개만 돈다.
FAIL 이면 여기서 멈추고 출력을 확인한다.


In [ ]:
!python -m lidarfly.tests


## ③ 빠른 스모크 (선택)

학습 루프가 도는지 수천 스텝만 확인. **건너뛰고 ④ 로 가도 된다.**
저장은 `/content/lidarfly_smoke` (드라이브 `v1` 과 분리 — 스모크 zip 을 `--resume auto` 가 집지 않음).


In [ ]:
!python -m lidarfly.train --maps "ifac:3,roboracer_0817:3,random:2" --eval-maps ifac,roboracer_0817 \
    --v-max {V_MAX} --timesteps 4000 --n-envs 2 --learning-starts 500 \
    --eval-freq 100000 --save-dir /content/lidarfly_smoke --device cuda


## ④ 본 학습

| 설정 | 의미 |
|--|--|
| `MAPS` | `ifac:3,roboracer_0817:3,random:2` — random 은 에피소드마다 만드는 폐곡선 |
| `EVAL_MAPS` | 5만 스텝마다 평가. `best_model.zip` 은 이 맵들 평균 점수 |
| `STEPS_PER_SESSION` | 이번 세션에 **추가로** 학습할 스텝 |
| `TIME_LIMIT_MIN` | 이 시간이 지나면 저장하고 종료 (무료 코랩은 몇 시간 뒤 끊김) |

- `last_model.zip` 은 10분마다, `checkpoints/` 는 10만 스텝마다 드라이브에 저장.
- 다시 이 셀을 실행하면 `--resume auto` 가 `v1` 안 최신 zip 에서 이어 간다. 버퍼는 비어 있으므로 처음 구간은 정책 행동으로 다시 모은다.
- TensorBoard 이벤트는 `{RUN_DIR}/tb`. 랩 기록은 `eval.csv`.


In [ ]:
MAPS = 'ifac:3,roboracer_0817:3,random:2'
EVAL_MAPS = 'ifac,roboracer_0817'
STEPS_PER_SESSION = 1_000_000
TIME_LIMIT_MIN = 150
N_ENVS = 4

!python -m lidarfly.train --maps "{MAPS}" --eval-maps {EVAL_MAPS} \
    --v-max {V_MAX} --timesteps {STEPS_PER_SESSION} --n-envs {N_ENVS} --subproc --device cuda \
    --time-limit-min {TIME_LIMIT_MIN} \
    --save-dir "{RUN_DIR}" --resume auto


## ⑤ 평가 — 맵별 랩타임·완주율

`best_model.zip` 이 생긴 뒤. 출발 4곳 × 3바퀴. 줄 끝 `[lidarfly eval]` 이 맵별 완주 수와 최고랩.


In [ ]:
!python -m lidarfly.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,roboracer_0817 --spawns 4 --laps 3 --obstacles 0 --v-max {V_MAX}
!python -m lidarfly.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,roboracer_0817 --spawns 4 --laps 3 --obstacles 2 --v-max {V_MAX}


## ⑥ TensorBoard

④ 가 한 번이라도 돌았으면 `{RUN_DIR}/tb` 에 SAC 손실·평가 스칼라가 쌓인다. 학습 셀과 따로 실행해도 된다.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"
